# Homework 5: Translation task (German → English)

[Qwen2.5-VL](https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct) is a
vision-language model, but it is also a strong **text** chat model. Here we use
it **text-only** (no images) to translate the German transcript from
`00_PrepData` into English.

We translate **segment by segment** so each request stays short and the
timestamps are preserved — this also scales to clips of any length.

**Sources**
- Qwen2.5-VL: <https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct> ([report](https://arxiv.org/abs/2502.13923))
- Model class `QwenChat` lives in `src/video_pipeline/qwen_text.py`.

In [1]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

# Make the vendored `video_pipeline` package importable even if the project
# was not installed (e.g. running `jupyter` outside `uv run`).
SRC = Path.cwd() / "src"
if SRC.exists() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

PREPARED = Path("prepared")
PREPARED.mkdir(exist_ok=True)

transcript = json.loads((PREPARED / "transcript.json").read_text(encoding="utf-8"))
print(f"loaded transcript: {len(transcript['segments'])} segments, "
      f"{len(transcript['text'])} chars")
print("\nfirst 300 chars (German):\n", transcript["text"][:300])

loaded transcript: 9 segments, 1698 chars

first 300 chars (German):
 Das italienische Universal-Genie Leonardo da Vinci hat nicht nur die Mona Lisa und andere schöne Bilder gemalt, er war auch ein genialer Ingenieur. Viele Skizzen und Beschreibungen seiner Erfindungen sind bis heute erhalten. Zahnräder spielten bei seinen Erfindungen eine große Rolle. Das Zahnrad an 


## Translate each segment with Qwen2.5-VL

`QwenChat.translate(text, source, target)` wraps a single system+user chat turn.
We load the model once (`with QwenChat() as qc:`) and reuse it across all
segments, then it is unloaded and the GPU cache is cleared on exit.

In [2]:
from video_pipeline.qwen_text import QwenChat

# TODO (Exercise 2a): translate every segment from German into English.
translated_segments = []
with QwenChat() as qc:
    for i, seg in enumerate(transcript["segments"], start=1):
        english = qc.translate(seg["text"], source="german", target="english")
        translated_segments.append({**seg, "text": english})
        print(f"[{i}/{len(transcript['segments'])}] {english[:70]}")

english_text = "\n".join(s["text"] for s in translated_segments)

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/57.6k [00:00<?, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

/home/azureuser/Projects/GAI4_course/HW_5/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

/home/azureuser/Projects/GAI4_course/HW_5/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[1/9] The Italian universal genius Leonardo da Vinci not only painted the Mo
[2/9] Or a heavy-duty forklift. Slightly modified, this corresponds to today
[3/9] He developed many weapons on behalf of his employers.
[4/9] For example, this organ gun. Quasi the first machine gun.
[5/9] And even a tank that, however, due to a construction defect could not 
[6/9] Bitte übersetzen Sie den Text aus dem Deutschen ins Englische.
[7/9] Subtitling for ZDF for funk, 2017. Quasi the first machine gun. And ev
[8/9] His design of a parachute. A success, albeit only after 500 years.
[9/9] The British Adrian Nicholas sailed through the air in 2000.


## Save a `TranslationResult`

We store the full English text in the project's `TranslationResult` schema so
the next notebook (and the CLI) can consume it.

In [3]:
from video_pipeline.models import TranslationResult

result = TranslationResult(
    source_path=str(PREPARED / "transcript.json"),
    source_language="German",
    target_language="English",
    source_text=transcript["text"],
    text=english_text,
)
out = PREPARED / "translation.json"
out.write_text(result.model_dump_json(indent=2), encoding="utf-8")
print("wrote", out, "-", len(english_text), "chars")

wrote prepared/translation.json - 1642 chars


In [4]:
# German vs English, side by side per segment.
for de, en in zip(transcript["segments"][:6], translated_segments[:6]):
    print(f"[{de['start']:6.1f}s] DE: {de['text']}")
    print(f"          EN: {en['text']}\n")

[   1.0s] DE: Das italienische Universal-Genie Leonardo da Vinci hat nicht nur die Mona Lisa und andere schöne Bilder gemalt, er war auch ein genialer Ingenieur. Viele Skizzen und Beschreibungen seiner Erfindungen sind bis heute erhalten. Zahnräder spielten bei seinen Erfindungen eine große Rolle. Das Zahnrad an sich existierte bereits, aber da Vinci entwickelte daraus zahlreiche Maschinen. Zum Beispiel das Kettenradgetriebe, das wir heute in jedem Fahrrad finden.
          EN: The Italian universal genius Leonardo da Vinci not only painted the Mona Lisa and other beautiful pictures, he was also a brilliant engineer. Many of his sketches and descriptions of his inventions have survived to this day. Gears played a significant role in his inventions. The gear itself already existed, but Da Vinci developed numerous machines from it. For example, the chain drive mechanism, which we find today in every bicycle.

[  28.2s] DE: Oder einen Schwerlastheber. Etwas abgewandelt entspricht das dem 

---
**Next:** `02_Summarize_OCR.ipynb` — summarise the English transcript and OCR the
on-screen text from the video frames.